# Adam: A Method for Stochastic Optimization — Implementation / 구현

**Paper**: Kingma, D. P. & Ba, J. L. "Adam: A Method for Stochastic Optimization", ICLR 2015. arXiv:1412.6980

이 노트북은 Adam optimizer를 NumPy로 처음부터 구현하고, 다른 optimizer들과 비교합니다.
This notebook implements the Adam optimizer from scratch using NumPy and compares it with other optimizers.

**구현 내용 / What we implement**:
1. SGD, Momentum, AdaGrad, RMSProp, Adam, AdaMax — 모든 optimizer를 처음부터 구현
2. Bias correction의 효과 시각화
3. 2D Rosenbrock 함수에서의 optimizer 궤적 비교
4. MNIST logistic regression에서의 학습 곡선 비교 (논문 Section 6.1 재현)
5. PyTorch의 Adam과 비교 검증

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from typing import Callable, Optional
from abc import ABC, abstractmethod

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True

np.random.seed(42)

## Part 1: Optimizer Zoo from Scratch / Optimizer 동물원 — 처음부터 구현

Adam을 이해하려면, Adam이 통합한 두 계열의 optimizer를 먼저 구현해야 합니다.
각 optimizer를 동일한 인터페이스로 구현하여 공정한 비교를 할 수 있게 합니다.

To understand Adam, we first implement the two lineages of optimizers that Adam unifies.
Each optimizer shares the same interface for fair comparison.

### Optimizer 계보 / Genealogy

| Lineage / 계열 | Optimizer | Update Rule |
|---|---|---|
| **Baseline** | SGD | $\theta_{t+1} = \theta_t - \alpha \cdot g_t$ |
| **Momentum** | SGD+Momentum | $v_t = \beta v_{t-1} + g_t$; $\theta_{t+1} = \theta_t - \alpha v_t$ |
| **Adaptive** | AdaGrad | $G_t = G_{t-1} + g_t^2$; $\theta_{t+1} = \theta_t - \alpha g_t / (\sqrt{G_t} + \epsilon)$ |
| **Adaptive** | RMSProp | $v_t = \beta v_{t-1} + (1-\beta) g_t^2$; $\theta_{t+1} = \theta_t - \alpha g_t / (\sqrt{v_t} + \epsilon)$ |
| **Both** | **Adam** | $m_t, v_t$ with bias correction |
| **Both** | **AdaMax** | $m_t$ with $L^\infty$ norm |

In [ ]:
class Optimizer(ABC):
    """Base class for all optimizers."""

    @abstractmethod
    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        """Perform a single optimization step.

        Args:
            params: Current parameter values.
            grads: Gradient of the loss w.r.t. params.

        Returns:
            Updated parameter values.
        """

    def reset(self) -> None:
        """Reset optimizer state (timestep, moment estimates, etc.)."""
        pass


class SGD(Optimizer):
    """Vanilla Stochastic Gradient Descent."""

    def __init__(self, lr: float = 0.01):
        self.lr = lr

    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        return params - self.lr * grads


class SGDMomentum(Optimizer):
    """SGD with classical momentum (Polyak, 1964)."""

    def __init__(self, lr: float = 0.01, beta: float = 0.9):
        self.lr = lr
        self.beta = beta
        self.v: Optional[np.ndarray] = None

    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        if self.v is None:
            self.v = np.zeros_like(params)
        self.v = self.beta * self.v + grads
        return params - self.lr * self.v

    def reset(self) -> None:
        self.v = None


class AdaGrad(Optimizer):
    """AdaGrad optimizer (Duchi et al., 2011).

    Accumulates squared gradients to adapt per-parameter learning rates.
    Problem: learning rate monotonically decreases.
    """

    def __init__(self, lr: float = 0.01, eps: float = 1e-8):
        self.lr = lr
        self.eps = eps
        self.G: Optional[np.ndarray] = None

    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        if self.G is None:
            self.G = np.zeros_like(params)
        self.G += grads ** 2
        return params - self.lr * grads / (np.sqrt(self.G) + self.eps)

    def reset(self) -> None:
        self.G = None


class RMSProp(Optimizer):
    """RMSProp optimizer (Tieleman & Hinton, 2012).

    Uses EMA of squared gradients instead of cumulative sum.
    Fixes AdaGrad's monotonically decreasing learning rate.
    No bias correction.
    """

    def __init__(self, lr: float = 0.001, beta: float = 0.999, eps: float = 1e-8):
        self.lr = lr
        self.beta = beta
        self.eps = eps
        self.v: Optional[np.ndarray] = None

    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        if self.v is None:
            self.v = np.zeros_like(params)
        self.v = self.beta * self.v + (1 - self.beta) * grads ** 2
        return params - self.lr * grads / (np.sqrt(self.v) + self.eps)

    def reset(self) -> None:
        self.v = None


class Adam(Optimizer):
    """Adam optimizer (Kingma & Ba, 2014).

    Combines momentum (1st moment) and RMSProp (2nd moment)
    with initialization bias correction.

    Algorithm 1 from the paper, reproduced exactly.
    """

    def __init__(
        self,
        lr: float = 0.001,
        beta1: float = 0.9,
        beta2: float = 0.999,
        eps: float = 1e-8,
        bias_correction: bool = True,
    ):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.bias_correction = bias_correction
        self.m: Optional[np.ndarray] = None
        self.v: Optional[np.ndarray] = None
        self.t = 0

    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        if self.m is None:
            self.m = np.zeros_like(params)
            self.v = np.zeros_like(params)

        self.t += 1

        # Update biased first moment estimate
        self.m = self.beta1 * self.m + (1 - self.beta1) * grads
        # Update biased second raw moment estimate
        self.v = self.beta2 * self.v + (1 - self.beta2) * grads ** 2

        if self.bias_correction:
            # Compute bias-corrected estimates
            m_hat = self.m / (1 - self.beta1 ** self.t)
            v_hat = self.v / (1 - self.beta2 ** self.t)
        else:
            m_hat = self.m
            v_hat = self.v

        return params - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

    def reset(self) -> None:
        self.m = None
        self.v = None
        self.t = 0


class AdaMax(Optimizer):
    """AdaMax optimizer (Kingma & Ba, 2014).

    Algorithm 2 from the paper: L-infinity norm variant of Adam.
    Uses max of exponentially weighted gradient magnitudes
    instead of L2 norm (second moment).
    """

    def __init__(
        self,
        lr: float = 0.002,
        beta1: float = 0.9,
        beta2: float = 0.999,
    ):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.m: Optional[np.ndarray] = None
        self.u: Optional[np.ndarray] = None
        self.t = 0

    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        if self.m is None:
            self.m = np.zeros_like(params)
            self.u = np.zeros_like(params)

        self.t += 1

        # Update biased first moment estimate
        self.m = self.beta1 * self.m + (1 - self.beta1) * grads
        # Update exponentially weighted infinity norm
        self.u = np.maximum(self.beta2 * self.u, np.abs(grads))

        # Bias correction only needed for first moment
        lr_t = self.lr / (1 - self.beta1 ** self.t)

        return params - lr_t * self.m / (self.u + 1e-8)

    def reset(self) -> None:
        self.m = None
        self.u = None
        self.t = 0


print("All optimizers defined successfully.")
print("SGD, SGDMomentum, AdaGrad, RMSProp, Adam, AdaMax")

## Part 2: Bias Correction 시각화 / Visualizing Bias Correction

Adam의 핵심 혁신인 bias correction의 효과를 직접 확인합니다.
$v_0 = 0$으로 초기화하면 초기 moment 추정이 실제보다 훨씬 작아집니다.
$(1 - \beta^t)$로 나누어 보정하면 초기부터 정확한 추정을 얻을 수 있습니다.

We directly visualize the effect of bias correction, Adam's key innovation.
Initializing $v_0 = 0$ makes early moment estimates much smaller than the true value.
Dividing by $(1 - \beta^t)$ corrects this from the very first timestep.

### 수학적 유도 / Mathematical Derivation

$$\mathbb{E}[v_t] = \mathbb{E}[g_t^2] \cdot (1 - \beta_2^t)$$

따라서 bias-corrected 추정: / So the bias-corrected estimate:

$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t} \quad \Rightarrow \quad \mathbb{E}[\hat{v}_t] = \mathbb{E}[g_t^2]$$

In [ ]:
def demonstrate_bias_correction(
    true_second_moment: float = 1.0,
    beta2: float = 0.999,
    n_steps: int = 2000,
    seed: int = 42,
) -> None:
    """Demonstrate the effect of bias correction on moment estimates.

    Simulates gradients drawn from N(0, true_second_moment) and shows
    how the raw EMA (biased) vs. bias-corrected EMA track the true value.

    Args:
        true_second_moment: True E[g^2] of the gradient distribution.
        beta2: Exponential decay rate for second moment.
        n_steps: Number of optimization steps to simulate.
        seed: Random seed for reproducibility.
    """
    rng = np.random.default_rng(seed)
    gradients = rng.normal(0, np.sqrt(true_second_moment), n_steps)

    v_biased = np.zeros(n_steps)
    v_corrected = np.zeros(n_steps)
    correction_factor = np.zeros(n_steps)

    v = 0.0
    for t in range(n_steps):
        v = beta2 * v + (1 - beta2) * gradients[t] ** 2
        v_biased[t] = v
        v_corrected[t] = v / (1 - beta2 ** (t + 1))
        correction_factor[t] = 1.0 / (1 - beta2 ** (t + 1))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Plot 1: Biased vs corrected estimates
    ax = axes[0]
    ax.plot(v_biased, label=f'Biased $v_t$ (raw EMA)', alpha=0.8)
    ax.plot(v_corrected, label=r'Corrected $\hat{v}_t = v_t/(1-\beta_2^t)$', alpha=0.8)
    ax.axhline(y=true_second_moment, color='k', linestyle='--',
               label=f'True $E[g^2]$ = {true_second_moment}')
    ax.set_xlabel('Timestep $t$')
    ax.set_ylabel('Second moment estimate')
    ax.set_title(f'Bias Correction Effect ($\\beta_2={beta2}$)')
    ax.legend()

    # Plot 2: Zoom into first 50 steps
    ax = axes[1]
    t_zoom = 50
    ax.plot(range(t_zoom), v_biased[:t_zoom],
            label='Biased $v_t$', marker='o', markersize=3)
    ax.plot(range(t_zoom), v_corrected[:t_zoom],
            label=r'Corrected $\hat{v}_t$', marker='s', markersize=3)
    ax.axhline(y=true_second_moment, color='k', linestyle='--',
               label=f'True $E[g^2]$')
    ax.set_xlabel('Timestep $t$')
    ax.set_ylabel('Second moment estimate')
    ax.set_title(f'First {t_zoom} Steps (Zoomed)')
    ax.legend()

    # Plot 3: Correction factor over time
    ax = axes[2]
    ax.plot(correction_factor, color='red')
    ax.set_xlabel('Timestep $t$')
    ax.set_ylabel('Correction factor $1/(1-\\beta_2^t)$')
    ax.set_title('Correction Factor Over Time')
    ax.set_yscale('log')

    # Annotate key values
    for t_val in [1, 10, 100, 1000]:
        if t_val <= n_steps:
            cf = 1.0 / (1 - beta2 ** t_val)
            ax.annotate(f't={t_val}: {cf:.1f}x',
                        xy=(t_val, cf), fontsize=9,
                        arrowprops=dict(arrowstyle='->', color='gray'),
                        xytext=(t_val + 100, cf * 1.5))

    plt.tight_layout()
    plt.show()


demonstrate_bias_correction(beta2=0.999)
demonstrate_bias_correction(beta2=0.99)

## Part 3: 2D Optimizer 궤적 비교 / 2D Optimizer Trajectory Comparison

Rosenbrock 함수 $f(x, y) = (a - x)^2 + b(y - x^2)^2$에서 각 optimizer의 궤적을 비교합니다.
이 함수는 좁고 긴 valley를 가지고 있어서 optimizer의 성능 차이를 극적으로 보여줍니다.

We compare optimizer trajectories on the Rosenbrock function $f(x, y) = (a - x)^2 + b(y - x^2)^2$.
This function has a narrow, curved valley that dramatically reveals performance differences between optimizers.

- 최적점은 $(a, a^2) = (1, 1)$
- Valley를 따라가야 하므로 momentum과 adaptive rate의 장점이 드러남
- SGD는 진동, AdaGrad는 정체, Adam은 부드럽게 수렴하는 것을 관찰

In [ ]:
def rosenbrock(xy: np.ndarray, a: float = 1.0, b: float = 100.0) -> float:
    """Compute Rosenbrock function value.

    Args:
        xy: 2D parameter vector [x, y].
        a: Parameter a (minimum at x=a).
        b: Parameter b (controls valley curvature).

    Returns:
        Function value f(x, y) = (a - x)^2 + b(y - x^2)^2.
    """
    x, y = xy
    return (a - x) ** 2 + b * (y - x ** 2) ** 2


def rosenbrock_grad(xy: np.ndarray, a: float = 1.0, b: float = 100.0) -> np.ndarray:
    """Compute gradient of the Rosenbrock function.

    Args:
        xy: 2D parameter vector [x, y].
        a: Parameter a.
        b: Parameter b.

    Returns:
        Gradient [df/dx, df/dy].
    """
    x, y = xy
    dfdx = -2 * (a - x) + 2 * b * (y - x ** 2) * (-2 * x)
    dfdy = 2 * b * (y - x ** 2)
    return np.array([dfdx, dfdy])


def optimize_2d(
    optimizer: Optimizer,
    grad_fn: Callable,
    x0: np.ndarray,
    n_steps: int = 5000,
) -> np.ndarray:
    """Run optimizer on a 2D function and record trajectory.

    Args:
        optimizer: Optimizer instance.
        grad_fn: Gradient function.
        x0: Initial point.
        n_steps: Number of optimization steps.

    Returns:
        Array of shape (n_steps+1, 2) with the trajectory.
    """
    optimizer.reset()
    trajectory = [x0.copy()]
    x = x0.copy()
    for _ in range(n_steps):
        g = grad_fn(x)
        x = optimizer.step(x, g)
        trajectory.append(x.copy())
    return np.array(trajectory)


# Define optimizers with tuned learning rates for Rosenbrock
optimizers = {
    'SGD': SGD(lr=0.0005),
    'Momentum': SGDMomentum(lr=0.0005, beta=0.9),
    'AdaGrad': AdaGrad(lr=0.1),
    'RMSProp': RMSProp(lr=0.001, beta=0.9),
    'Adam': Adam(lr=0.005),
    'AdaMax': AdaMax(lr=0.005),
}

x0 = np.array([-1.5, 2.0])
n_steps = 5000
trajectories = {}

for name, opt in optimizers.items():
    traj = optimize_2d(opt, rosenbrock_grad, x0, n_steps)
    trajectories[name] = traj

# Plot
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Create contour grid
x_range = np.linspace(-2, 2, 300)
y_range = np.linspace(-1, 3, 300)
X, Y = np.meshgrid(x_range, y_range)
Z = (1 - X) ** 2 + 100 * (Y - X ** 2) ** 2

colors = {
    'SGD': 'gray', 'Momentum': 'blue', 'AdaGrad': 'green',
    'RMSProp': 'orange', 'Adam': 'red', 'AdaMax': 'purple',
}

# Left: Trajectories on contour
ax = axes[0]
ax.contour(X, Y, Z, levels=np.logspace(-1, 3.5, 20), cmap='viridis', alpha=0.3)
for name, traj in trajectories.items():
    ax.plot(traj[:, 0], traj[:, 1], '-', color=colors[name],
            label=name, alpha=0.7, linewidth=1.5)
ax.plot(1, 1, 'k*', markersize=15, zorder=10, label='Optimum (1,1)')
ax.plot(x0[0], x0[1], 'ko', markersize=8, zorder=10)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Optimizer Trajectories on Rosenbrock Function')
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim(-2, 2)
ax.set_ylim(-1, 3)

# Right: Loss over iterations
ax = axes[1]
for name, traj in trajectories.items():
    losses = [rosenbrock(traj[i]) for i in range(len(traj))]
    ax.plot(losses, color=colors[name], label=name, alpha=0.8)
ax.set_xlabel('Iteration')
ax.set_ylabel('$f(x, y)$ (log scale)')
ax.set_title('Convergence: Rosenbrock Function')
ax.set_yscale('log')
ax.legend()

plt.tight_layout()
plt.show()

# Print final distances to optimum
print("\nFinal distance to optimum (1, 1):")
print("-" * 40)
for name, traj in trajectories.items():
    dist = np.linalg.norm(traj[-1] - np.array([1.0, 1.0]))
    loss = rosenbrock(traj[-1])
    print(f"{name:12s}: dist = {dist:.6f}, loss = {loss:.6e}")

## Part 4: Adam vs Adam (no bias correction) / Bias Correction의 실제 효과

Bias correction 유무에 따른 Adam의 학습 차이를 비교합니다. 특히 $\beta_2$가 1에 가까울 때 차이가 극적입니다.
논문 Section 6.4의 핵심 실험을 재현합니다: bias correction 없는 Adam ≈ RMSProp with momentum.

We compare Adam with and without bias correction. The difference is dramatic when $\beta_2$ is close to 1.
This reproduces the key experiment from Section 6.4: Adam without bias correction ≈ RMSProp with momentum.

In [ ]:
def noisy_quadratic_loss(
    theta: np.ndarray,
    rng: np.random.Generator,
    noise_std: float = 0.5,
) -> tuple[float, np.ndarray]:
    """Compute noisy quadratic loss and gradient.

    A simple stochastic objective: f(theta) = 0.5 * ||theta||^2 + noise.

    Args:
        theta: Parameter vector.
        rng: Random number generator.
        noise_std: Standard deviation of gradient noise.

    Returns:
        Tuple of (loss, gradient).
    """
    noise = rng.normal(0, noise_std, theta.shape)
    grad = theta + noise
    loss = 0.5 * np.sum(theta ** 2)
    return loss, grad


beta2_values = [0.99, 0.999, 0.9999]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, beta2 in enumerate(beta2_values):
    ax = axes[idx]
    n_steps = 3000
    dim = 10

    for bc, label, color, ls in [
        (True, 'Adam (with bias correction)', 'red', '-'),
        (False, 'Adam (no bias correction)', 'green', '--'),
    ]:
        rng = np.random.default_rng(42)
        theta = rng.normal(0, 1, dim)
        opt = Adam(lr=0.01, beta1=0.9, beta2=beta2, bias_correction=bc)
        losses = []

        for _ in range(n_steps):
            loss, grad = noisy_quadratic_loss(theta, rng)
            losses.append(loss)
            theta = opt.step(theta, grad)

        ax.plot(losses, color=color, linestyle=ls, label=label, alpha=0.8)

    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss (log scale)')
    ax.set_title(f'$\\beta_2 = {beta2}$')
    ax.set_yscale('log')
    ax.legend(fontsize=8)

plt.suptitle('Effect of Bias Correction for Different $\\beta_2$ Values',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Key observation: As β₂ → 1, the gap between corrected and uncorrected grows.")
print("At β₂=0.9999, uncorrected Adam shows significant instability in early training.")

## Part 5: MNIST Logistic Regression 실험 재현 / Reproducing Paper's Section 6.1

논문 Figure 1(왼쪽)을 재현합니다: MNIST에서 L2-regularized multi-class logistic regression.
모든 optimizer를 동일한 조건에서 비교합니다.

We reproduce Figure 1 (left) from the paper: L2-regularized multi-class logistic regression on MNIST.
All optimizers are compared under the same conditions.

### 모델 / Model

Softmax regression (multinomial logistic regression):

$$P(y=k | x) = \frac{\exp(W_k^T x + b_k)}{\sum_{j} \exp(W_j^T x + b_j)}$$

$$\text{Loss} = -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i | x_i) + \frac{\lambda}{2} \|W\|_F^2$$

In [ ]:
def load_mnist() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Load MNIST dataset using scikit-learn.

    Returns:
        Tuple of (X_train, y_train, X_test, y_test) with normalized pixel values.
    """
    from sklearn.datasets import fetch_openml

    mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
    X, y = mnist.data.astype(np.float32), mnist.target.astype(np.int64)

    # Normalize to [0, 1]
    X = X / 255.0

    # Train/test split (60k/10k)
    X_train, X_test = X[:60000], X[60000:]
    y_train, y_test = y[:60000], y[60000:]

    return X_train, y_train, X_test, y_test


def softmax(logits: np.ndarray) -> np.ndarray:
    """Numerically stable softmax.

    Args:
        logits: Array of shape (batch_size, n_classes).

    Returns:
        Probability array of the same shape.
    """
    exp_logits = np.exp(logits - logits.max(axis=1, keepdims=True))
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def cross_entropy_loss_and_grad(
    params: np.ndarray,
    X_batch: np.ndarray,
    y_batch: np.ndarray,
    n_classes: int = 10,
    l2_reg: float = 1e-4,
) -> tuple[float, np.ndarray]:
    """Compute cross-entropy loss and gradient for softmax regression.

    Args:
        params: Flattened parameter vector [W (784*10), b (10)].
        X_batch: Input features of shape (batch_size, 784).
        y_batch: Labels of shape (batch_size,).
        n_classes: Number of output classes.
        l2_reg: L2 regularization strength.

    Returns:
        Tuple of (loss, gradient) where gradient has same shape as params.
    """
    n_features = X_batch.shape[1]
    W = params[:n_features * n_classes].reshape(n_features, n_classes)
    b = params[n_features * n_classes:]

    batch_size = X_batch.shape[0]
    logits = X_batch @ W + b  # (batch_size, n_classes)
    probs = softmax(logits)

    # One-hot encode labels
    one_hot = np.zeros_like(probs)
    one_hot[np.arange(batch_size), y_batch] = 1

    # Cross-entropy loss + L2 regularization
    loss = -np.mean(np.sum(one_hot * np.log(probs + 1e-12), axis=1))
    loss += 0.5 * l2_reg * np.sum(W ** 2)

    # Gradients
    d_logits = (probs - one_hot) / batch_size  # (batch_size, n_classes)
    dW = X_batch.T @ d_logits + l2_reg * W  # (n_features, n_classes)
    db = d_logits.sum(axis=0)  # (n_classes,)

    grad = np.concatenate([dW.ravel(), db])
    return loss, grad


print("Loading MNIST...")
X_train, y_train, X_test, y_test = load_mnist()
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Labels: {np.unique(y_train)}")

In [ ]:
def train_logistic_regression(
    optimizer: Optimizer,
    X_train: np.ndarray,
    y_train: np.ndarray,
    n_epochs: int = 30,
    batch_size: int = 128,
    seed: int = 42,
) -> list[float]:
    """Train logistic regression with the given optimizer.

    Following the paper's Section 6.1 setup:
    - L2-regularized multi-class logistic regression
    - Mini-batch size 128
    - 784-dimensional input (MNIST)

    Args:
        optimizer: Optimizer instance.
        X_train: Training features.
        y_train: Training labels.
        n_epochs: Number of full passes over the data.
        batch_size: Mini-batch size.
        seed: Random seed.

    Returns:
        List of training losses (one per mini-batch).
    """
    rng = np.random.default_rng(seed)
    n_features = X_train.shape[1]
    n_classes = 10

    # Initialize parameters
    params = np.zeros(n_features * n_classes + n_classes)
    optimizer.reset()

    losses = []
    n_samples = X_train.shape[0]

    for epoch in range(n_epochs):
        indices = rng.permutation(n_samples)
        for start in range(0, n_samples, batch_size):
            batch_idx = indices[start:start + batch_size]
            X_batch = X_train[batch_idx]
            y_batch = y_train[batch_idx]

            loss, grad = cross_entropy_loss_and_grad(params, X_batch, y_batch)
            params = optimizer.step(params, grad)
            losses.append(loss)

    return losses


# Paper uses alpha_t = alpha / sqrt(t) for logistic regression
# We use constant lr for simplicity, tuned per optimizer
mnist_optimizers = {
    'SGD': SGD(lr=0.5),
    'Momentum': SGDMomentum(lr=0.1, beta=0.9),
    'AdaGrad': AdaGrad(lr=0.01),
    'RMSProp': RMSProp(lr=0.001, beta=0.9),
    'Adam': Adam(lr=0.001),
    'AdaMax': AdaMax(lr=0.002),
}

n_epochs = 30
results = {}

for name, opt in mnist_optimizers.items():
    print(f"Training with {name}...")
    losses = train_logistic_regression(opt, X_train, y_train, n_epochs=n_epochs)
    results[name] = losses
    print(f"  Final loss: {losses[-1]:.4f}")

# Plot: reproducing Figure 1 (left) from the paper
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {
    'SGD': 'gray', 'Momentum': 'blue', 'AdaGrad': 'green',
    'RMSProp': 'orange', 'Adam': 'red', 'AdaMax': 'purple',
}

# Left: Training loss over iterations
ax = axes[0]
iters_per_epoch = len(X_train) // 128
for name, losses in results.items():
    # Smooth with running average for readability
    window = iters_per_epoch // 2
    smoothed = np.convolve(losses, np.ones(window) / window, mode='valid')
    x_axis = np.arange(len(smoothed)) / iters_per_epoch
    ax.plot(x_axis, smoothed, color=colors[name], label=name, alpha=0.8)
ax.set_xlabel('Epochs')
ax.set_ylabel('Training Loss (NLL)')
ax.set_title('MNIST Logistic Regression — Training Loss\n(cf. Paper Figure 1, left)')
ax.legend()

# Right: Same with log scale
ax = axes[1]
for name, losses in results.items():
    window = iters_per_epoch // 2
    smoothed = np.convolve(losses, np.ones(window) / window, mode='valid')
    x_axis = np.arange(len(smoothed)) / iters_per_epoch
    ax.plot(x_axis, smoothed, color=colors[name], label=name, alpha=0.8)
ax.set_xlabel('Epochs')
ax.set_ylabel('Training Loss (log scale)')
ax.set_title('MNIST Logistic Regression — Log Scale')
ax.set_yscale('log')
ax.legend()

plt.tight_layout()
plt.show()

## Part 6: PyTorch 비교 검증 / Verification Against PyTorch

우리가 구현한 Adam이 PyTorch의 `torch.optim.Adam`과 동일한 결과를 내는지 검증합니다.
동일한 초기 파라미터, 동일한 gradient를 주어 step-by-step으로 비교합니다.

We verify our Adam implementation produces identical results to PyTorch's `torch.optim.Adam`.
Given identical initial parameters and gradients, we compare step by step.

In [ ]:
import torch


def verify_adam_against_pytorch(
    n_steps: int = 100,
    dim: int = 10,
    lr: float = 0.001,
    beta1: float = 0.9,
    beta2: float = 0.999,
    eps: float = 1e-8,
    seed: int = 42,
) -> None:
    """Compare our Adam with PyTorch's implementation step by step.

    Args:
        n_steps: Number of optimization steps.
        dim: Parameter dimension.
        lr: Learning rate.
        beta1: First moment decay rate.
        beta2: Second moment decay rate.
        eps: Numerical stability constant.
        seed: Random seed.
    """
    rng = np.random.default_rng(seed)

    # Generate random gradients
    gradients = rng.normal(0, 1, (n_steps, dim)).astype(np.float64)

    # Initialize with same parameters
    init_params = rng.normal(0, 1, dim).astype(np.float64)

    # --- Our implementation ---
    our_adam = Adam(lr=lr, beta1=beta1, beta2=beta2, eps=eps)
    our_params = init_params.copy()
    our_trajectory = [our_params.copy()]

    for t in range(n_steps):
        our_params = our_adam.step(our_params, gradients[t])
        our_trajectory.append(our_params.copy())

    # --- PyTorch implementation ---
    pt_params = torch.tensor(init_params.copy(), dtype=torch.float64,
                             requires_grad=False)
    # PyTorch requires a Parameter to optimize
    pt_param = torch.nn.Parameter(pt_params.clone())
    pt_optimizer = torch.optim.Adam([pt_param], lr=lr, betas=(beta1, beta2), eps=eps)
    pt_trajectory = [pt_param.data.numpy().copy()]

    for t in range(n_steps):
        pt_optimizer.zero_grad()
        pt_param.grad = torch.tensor(gradients[t], dtype=torch.float64)
        pt_optimizer.step()
        pt_trajectory.append(pt_param.data.numpy().copy())

    # Compare
    our_trajectory = np.array(our_trajectory)
    pt_trajectory = np.array(pt_trajectory)

    max_diff = np.max(np.abs(our_trajectory - pt_trajectory))
    mean_diff = np.mean(np.abs(our_trajectory - pt_trajectory))

    print(f"Comparison over {n_steps} steps, dim={dim}")
    print(f"  Max absolute difference:  {max_diff:.2e}")
    print(f"  Mean absolute difference: {mean_diff:.2e}")
    print(f"  Match: {'YES' if max_diff < 1e-10 else 'CLOSE' if max_diff < 1e-6 else 'NO'}")

    # Plot first dimension trajectory
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.plot(our_trajectory[:, 0], label='Our Adam', linewidth=2)
    ax.plot(pt_trajectory[:, 0], '--', label='PyTorch Adam', linewidth=2)
    ax.set_xlabel('Step')
    ax.set_ylabel('Parameter value (dim 0)')
    ax.set_title('Parameter Trajectory Comparison')
    ax.legend()

    ax = axes[1]
    diffs = np.abs(our_trajectory - pt_trajectory).max(axis=1)
    ax.plot(diffs)
    ax.set_xlabel('Step')
    ax.set_ylabel('Max absolute difference')
    ax.set_title('Step-by-Step Difference')
    ax.set_yscale('log')

    plt.tight_layout()
    plt.show()


verify_adam_against_pytorch()

## Part 7: Effective Step Size와 SNR / Effective Step Size and SNR

Adam의 핵심 성질을 실험적으로 확인합니다:
- **Effective step size** $|\Delta_t| \lessapprox \alpha$ (trust region 성질)
- **SNR** $= |\hat{m}_t| / \sqrt{\hat{v}_t}$이 최적점 근처에서 감소 (automatic annealing)

We experimentally verify Adam's key properties:
- **Effective step size** $|\Delta_t| \lessapprox \alpha$ (trust region property)
- **SNR** $= |\hat{m}_t| / \sqrt{\hat{v}_t}$ decreases near optima (automatic annealing)

In [ ]:
def track_adam_internals(
    n_steps: int = 3000,
    dim: int = 10,
    lr: float = 0.01,
    seed: int = 42,
) -> None:
    """Track Adam's internal state to visualize trust region and SNR.

    Optimizes a noisy quadratic and records effective step sizes,
    SNR values, and loss to demonstrate automatic annealing.

    Args:
        n_steps: Number of optimization steps.
        dim: Parameter dimension.
        lr: Learning rate (alpha).
        seed: Random seed.
    """
    rng = np.random.default_rng(seed)
    theta = rng.normal(0, 3, dim)

    m = np.zeros(dim)
    v = np.zeros(dim)
    beta1, beta2, eps = 0.9, 0.999, 1e-8

    step_sizes = []
    snr_values = []
    losses = []

    for t in range(1, n_steps + 1):
        noise = rng.normal(0, 0.5, dim)
        g = theta + noise
        loss = 0.5 * np.sum(theta ** 2)
        losses.append(loss)

        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g ** 2

        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)

        # Effective step size per dimension
        delta = lr * m_hat / (np.sqrt(v_hat) + eps)
        step_sizes.append(np.abs(delta).mean())

        # SNR per dimension
        snr = np.abs(m_hat) / (np.sqrt(v_hat) + eps)
        snr_values.append(snr.mean())

        theta = theta - delta

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Plot 1: Effective step size vs alpha
    ax = axes[0]
    ax.plot(step_sizes, alpha=0.6, linewidth=0.8)
    ax.axhline(y=lr, color='red', linestyle='--', linewidth=2,
               label=f'$\\alpha$ = {lr}')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Mean $|\\Delta_t|$')
    ax.set_title('Effective Step Size (bounded by $\\alpha$)')
    ax.legend()

    # Plot 2: SNR (automatic annealing)
    ax = axes[1]
    ax.plot(snr_values, alpha=0.6, linewidth=0.8, color='green')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Mean SNR = $|\\hat{m}_t| / \\sqrt{\\hat{v}_t}$')
    ax.set_title('Signal-to-Noise Ratio (Automatic Annealing)')

    # Plot 3: Loss
    ax = axes[2]
    ax.plot(losses, alpha=0.6, linewidth=0.8, color='purple')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.set_title('Loss Over Time')
    ax.set_yscale('log')

    plt.suptitle(f'Adam Internals: Trust Region & SNR ($\\alpha$={lr}, dim={dim})',
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    print(f"Initial step size: {step_sizes[0]:.6f} (alpha = {lr})")
    print(f"Final step size:   {step_sizes[-1]:.6f}")
    print(f"Initial SNR:       {snr_values[0]:.6f}")
    print(f"Final SNR:         {snr_values[-1]:.6f}")
    print(f"SNR decreased by {snr_values[0]/snr_values[-1]:.1f}x → automatic annealing!")


track_adam_internals()

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| **Adam optimizer** | Algorithm 1: $m_t$, $v_t$ with bias correction | `torch.optim.Adam`, `tf.keras.optimizers.Adam` |
| **AdaMax optimizer** | Algorithm 2: $L^\infty$ norm variant | `torch.optim.Adamax` |
| **Bias correction** | $\hat{m}_t = m_t/(1-\beta_1^t)$ | All modern Adam implementations include this |
| **Weight decay** | L2 regularization in loss | **AdamW** (`torch.optim.AdamW`) — decoupled weight decay |
| **Learning rate warmup** | Not in paper (manual schedule) | **RAdam** — automatic warmup via variance rectification |
| **Convergence fix** | $O(\sqrt{T})$ regret in convex setting | **AMSGrad** — fixes non-convergence counterexamples |

### 다음 논문과의 연결 / Connection to Next Papers

이 논문에서 제안된 Adam은 딥러닝의 기본 optimizer로 자리잡았으며,
이후 논문들(AdamW, RAdam, LAMB, Lion 등)은 모두 Adam의 한계를 보완하는 방향으로 발전했습니다.
특히 AdamW는 weight decay의 올바른 적용법을 제시하여 현재 가장 널리 사용되는 Adam 변형입니다.

Adam proposed in this paper became the default optimizer for deep learning.
Subsequent papers (AdamW, RAdam, LAMB, Lion, etc.) all evolved by addressing Adam's limitations.
AdamW in particular presented the correct way to apply weight decay and is currently the most widely used Adam variant.